# Reinforcement Learning — Implementations

Value iteration, tabular Q-learning and a small DQN, each once more on tensors. The equivalence fixtures rebuild the notebook's GridWorld as demo tooling (the environment is scenery, not a step of any algorithm), route every random draw through the same numpy generator in every lane, and use fixed budgets so no lane stops early.

## 18_value_iteration

Dynamic programming: sweep Bellman backups until the value function stops moving. There is no gradient anywhere and nothing to fit, so there is no library lane — no library in this sandbox ships a generic MDP planner — and the torch lane is about tensorising, not differentiating.

### torch

The four nested Python loops become two tensor ops per sweep: gather next-state values, add rewards, max over actions. Synchronous sweeps replace the scratch lane's in-place ones — both contract to the same unique fixed point, exactly. **What torch adds:** the whole state–action space updated as one `R + γ·V[NS]` expression, which is also how you would batch this on a GPU.

In [ ]:
import numpy as np
import torch

# hints:
# 1. Precompute NS[s,a] and R[s,a] once; the transition loops run before iterating.
# 2. One whole sweep is Q = R + gamma * V[NS]; then V_new = Q.max(dim=1).values.
# 3. Pin V[goal] back to 0 after every sweep — the scratch loop simply skips the goal.
# 4. In-place (Gauss-Seidel) and synchronous sweeps share the same unique fixed point.


def value_iteration(env, gamma=0.99, theta=1e-6):
    """Synchronous Bellman backups on tensors: a sweep is two vectorised ops."""
    states = env.get_all_states()
    idx = {s: i for i, s in enumerate(states)}
    n = len(states)
    NS = torch.zeros((n, 4), dtype=torch.long)      # next-state index per (s, a)
    R = torch.zeros((n, 4), dtype=torch.float64)    # reward per (s, a)
    for s in states:
        for a in range(4):
            (prob, next_s, reward, done), = env.get_transitions(s, a)
            NS[idx[s], a] = idx[next_s]
            R[idx[s], a] = reward
    goal = idx[env.goal]
    V = torch.zeros(n, dtype=torch.float64)
    while True:
        Q = R + gamma * V[NS]                       # every (s, a) backup at once
        V_new = Q.max(dim=1).values
        V_new[goal] = 0.0                           # the goal is absorbing at value 0
        delta = float(torch.max(torch.abs(V_new - V)))
        V = V_new
        if delta < theta:
            break
    Q = R + gamma * V[NS]
    Q[goal] = 0.0
    V_dict = {s: float(V[idx[s]]) for s in states}
    Q_dict = {s: Q[idx[s]].numpy() for s in states}
    return V_dict, Q_dict


In [ ]:
# exports: V_table, Q_table
class _GridWorld:
    """Faithful copy of the notebook's GridWorld — the env is a fixture, not a step."""
    def __init__(self, rows=4, cols=4, start=(0, 0), goal=(3, 3),
                 obstacles=((1, 1), (2, 1))):
        self.rows, self.cols, self.start, self.goal = rows, cols, start, goal
        self.obstacles = list(obstacles)
        self.actions = [(-1, 0), (0, 1), (1, 0), (0, -1)]  # Up, Right, Down, Left
        self.reset()

    def reset(self):
        self.current_state = self.start
        return self.current_state

    def step(self, action_idx):
        if self.current_state == self.goal:
            return self.current_state, 0.0, True
        dr, dc = self.actions[action_idx]
        r, c = self.current_state
        nr, nc = r + dr, c + dc
        if 0 <= nr < self.rows and 0 <= nc < self.cols and (nr, nc) not in self.obstacles:
            self.current_state = (nr, nc)
        done = self.current_state == self.goal
        return self.current_state, 10.0 if done else -1.0, done

    def get_all_states(self):
        return [(r, c) for r in range(self.rows) for c in range(self.cols)
                if (r, c) not in self.obstacles]

    def get_transitions(self, state, action_idx):
        if state == self.goal:
            return [(1.0, state, 0.0, True)]
        dr, dc = self.actions[action_idx]
        r, c = state
        nr, nc = r + dr, c + dc
        ns = (nr, nc) if (0 <= nr < self.rows and 0 <= nc < self.cols
                          and (nr, nc) not in self.obstacles) else state
        done = ns == self.goal
        return [(1.0, ns, 10.0 if done else -1.0, done)]

_env_eq = _GridWorld()
_V_eq, _Q_eq = value_iteration(_env_eq, gamma=0.99, theta=1e-10)
V_table = [float(_V_eq[_s]) for _s in _env_eq.get_all_states()]
Q_table = [[float(_x) for _x in _Q_eq[_s]] for _s in _env_eq.get_all_states()]
print("V(start) =", round(V_table[0], 6))


In [ ]:
# The fixed point satisfies Bellman optimality: V(s) = max_a [r + γ V(s')].
_worst = 0.0
for _s in _env_eq.get_all_states():
    if _s == _env_eq.goal:
        continue
    _vals = []
    for _a in range(4):
        (_p, _s2, _r, _d), = _env_eq.get_transitions(_s, _a)
        _vals.append(_r + 0.99 * _V_eq[_s2])
    _worst = max(_worst, abs(_V_eq[_s] - max(_vals)))
assert _worst < 1e-9, "Bellman residual should vanish at the fixed point"

# The shortest path is 6 moves: 5 step penalties, then the +10 goal reward.
_v_direct = sum(-(0.99 ** _k) for _k in range(5)) + 10.0 * 0.99 ** 5
assert abs(_V_eq[(0, 0)] - _v_direct) < 1e-9, "V(start) equals the discounted 6-move return"

# The greedy policy read off Q walks that shortest path.
_pos = _env_eq.reset()
_n = 0
while _pos != _env_eq.goal and _n < 10:
    _pos, _, _ = _env_eq.step(int(np.argmax(_Q_eq[_pos])))
    _n += 1
assert _pos == _env_eq.goal and _n == 6, "greedy on Q reaches the goal in exactly 6 moves"

assert _V_eq[_env_eq.goal] == 0.0 and float(np.max(np.abs(_Q_eq[_env_eq.goal]))) == 0.0, \
    "the absorbing goal keeps value 0"


## 18_q_learning

Model-free TD control: learn Q from single sampled transitions. The update is differentiable (it is a semi-gradient step), so a torch lane exists; there is no library lane because neither sklearn nor torch ships a tabular Q-learning estimator worth comparing against.

### torch

The Q table becomes one tensor and the update rule is *derived* rather than transcribed: autograd differentiates ½(target − Q[s,a])² with the target detached, and one `SGD(lr=α)` step reproduces the scratch rule bit for bit. Exploration draws come from the same numpy generator in the same order, so both lanes see identical episodes. **What torch adds:** the TD update revealed as gradient descent on the squared TD error — the exact bridge to DQN below.

In [ ]:
import numpy as np
import torch

# hints:
# 1. Q[s,a] += α·(target − Q[s,a]) is one SGD step on the loss ½(target − Q[s,a])².
# 2. Compute the TD target under no_grad: it is data, not a thing to differentiate.
# 3. Draw exploration from the SAME numpy rng, in the same order, as the scratch agent.
# 4. SGD(lr=α) on that loss reproduces the tabular update to the last bit — check it.
# 5. Tie-breaking compares float64 for equality, so the tables must match exactly.


class QLearningAgent:
    """Tabular Q-learning with the Q table as one tensor. The TD update is not
    copied — it is *recovered*: autograd differentiates ½(target − Q[s,a])² and
    one SGD step at lr=α lands on exactly the scratch rule."""

    def __init__(self, env, alpha=0.1, gamma=0.99, epsilon=0.1):
        self.env = env
        self.alpha = alpha
        self.gamma = gamma
        self.epsilon = epsilon
        self.states = env.get_all_states()
        self.idx = {s: i for i, s in enumerate(self.states)}
        self.Q = torch.zeros((len(self.states), 4), dtype=torch.float64,
                             requires_grad=True)
        self.opt = torch.optim.SGD([self.Q], lr=alpha)

    def get_action(self, state):
        if rng.random() < self.epsilon:
            return rng.integers(0, 4)
        # Same tie-breaking draws as the scratch agent, on the same numbers.
        q_values = self.Q[self.idx[state]].detach().numpy()
        max_q = np.max(q_values)
        return rng.choice(np.where(q_values == max_q)[0])

    def update(self, state, action, reward, next_state, done):
        best_next_q = 0.0 if done else float(self.Q[self.idx[next_state]].detach().max())
        td_target = reward + self.gamma * best_next_q
        self.opt.zero_grad()
        q_sa = self.Q[self.idx[state], action]
        loss = 0.5 * (td_target - q_sa) ** 2
        loss.backward()          # dloss/dQ[s,a] = -(td_target - Q[s,a]) = -td_error
        self.opt.step()          # Q[s,a] -= α · grad  ==  Q[s,a] += α · td_error


In [ ]:
# exports: Q_table
class _GridWorld:
    """Faithful copy of the notebook's GridWorld — the env is a fixture, not a step."""
    def __init__(self, rows=4, cols=4, start=(0, 0), goal=(3, 3),
                 obstacles=((1, 1), (2, 1))):
        self.rows, self.cols, self.start, self.goal = rows, cols, start, goal
        self.obstacles = list(obstacles)
        self.actions = [(-1, 0), (0, 1), (1, 0), (0, -1)]  # Up, Right, Down, Left
        self.reset()

    def reset(self):
        self.current_state = self.start
        return self.current_state

    def step(self, action_idx):
        if self.current_state == self.goal:
            return self.current_state, 0.0, True
        dr, dc = self.actions[action_idx]
        r, c = self.current_state
        nr, nc = r + dr, c + dc
        if 0 <= nr < self.rows and 0 <= nc < self.cols and (nr, nc) not in self.obstacles:
            self.current_state = (nr, nc)
        done = self.current_state == self.goal
        return self.current_state, 10.0 if done else -1.0, done

    def get_all_states(self):
        return [(r, c) for r in range(self.rows) for c in range(self.cols)
                if (r, c) not in self.obstacles]

    def get_transitions(self, state, action_idx):
        if state == self.goal:
            return [(1.0, state, 0.0, True)]
        dr, dc = self.actions[action_idx]
        r, c = state
        nr, nc = r + dr, c + dc
        ns = (nr, nc) if (0 <= nr < self.rows and 0 <= nc < self.cols
                          and (nr, nc) not in self.obstacles) else state
        done = ns == self.goal
        return [(1.0, ns, 10.0 if done else -1.0, done)]

rng = np.random.default_rng(321)
_env_eq = _GridWorld()
_agent_eq = QLearningAgent(_env_eq, alpha=0.1, gamma=0.99, epsilon=0.1)
for _ep in range(150):
    _s = _env_eq.reset()
    _done = False
    _t = 0
    while not _done and _t < 250:
        _a = _agent_eq.get_action(_s)
        _s2, _r, _done = _env_eq.step(_a)
        _agent_eq.update(_s, _a, _r, _s2, _done)
        _s = _s2
        _t += 1
Q_table = [[float(_x) for _x in _agent_eq.Q[_agent_eq.idx[_s]].detach().numpy()]
           for _s in _env_eq.get_all_states()]
print("max Q =", round(max(max(_row) for _row in Q_table), 4))


In [ ]:
# Autograd + SGD really is the tabular rule: probe one hand-computed update.
_probe = QLearningAgent(_env_eq, alpha=0.1, gamma=0.99, epsilon=0.1)
with torch.no_grad():
    _probe.Q[_probe.idx[(0, 1)], 2] = 1.25
    _probe.Q[_probe.idx[(0, 2)], 0] = 2.0
_probe.update((0, 1), 2, -1.0, (0, 2), False)
_expected = 1.25 + 0.1 * ((-1.0 + 0.99 * 2.0) - 1.25)
assert abs(float(_probe.Q[_probe.idx[(0, 1)], 2]) - _expected) < 1e-12, \
    "one SGD step equals the hand TD update"

# The learned greedy policy solves the maze (shortest possible is 6 moves).
_pos = _env_eq.reset()
_n = 0
while _pos != _env_eq.goal and _n < 20:
    _pos, _, _ = _env_eq.step(int(np.argmax(_agent_eq.Q[_agent_eq.idx[_pos]].detach().numpy())))
    _n += 1
assert _pos == _env_eq.goal and _n >= 6, "greedy policy reaches the goal, no faster than 6"

# Targets never exceed the goal reward, so Q stays below it — an invariant.
assert float(_agent_eq.Q.detach().max()) <= 10.0 + 1e-9, "Q is bounded by the goal reward"

# The goal state is never acted from, so its row is untouched.
assert float(_agent_eq.Q[_agent_eq.idx[_env_eq.goal]].detach().abs().max()) == 0.0, \
    "the absorbing goal row stays zero"


## 18_dqn

Q-learning with the table replaced by a small MLP, plus the two DQN stabilisers: a replay buffer and a frozen target network. The equivalence fixture keeps the behaviour policy purely random (ε = 1) so that no action ever depends on network arithmetic — Q-learning is off-policy, so the nets still learn greedy values from that experience while every lane replays the identical trajectory.

### torch

The scratch network with its backward pass deleted: same numpy weight init, same replay draws, `loss.backward()` instead of forty lines of hand chain rule, `p -= lr·grad` under `no_grad`. **What torch adds:** autograd replaces the entire hand-written backward pass, and a check confirms the two gradients are the same numbers.

In [ ]:
import numpy as np
import torch

# hints:
# 1. Draw W1 and W2 with numpy's rng, then torch.as_tensor — identical init to scratch.
# 2. loss = ½·mean((q[i, aᵢ] − yᵢ)²): autograd's dq is the scratch (q − y)/N, exactly.
# 3. Clear grads (p.grad = None) before backward; step p -= lr·p.grad under no_grad.
# 4. Stay float64 end to end; the float32 one-hots are promoted exactly on conversion.


def state_to_onehot(state, rows=4, cols=4):
    idx = state[0] * cols + state[1]
    onehot = np.zeros(rows * cols, dtype=np.float32)
    onehot[idx] = 1.0
    return onehot

class ReplayBuffer:
    def __init__(self, capacity=2000):
        self.capacity = capacity
        self.buffer = []
        self.ptr = 0

    def push(self, state, action, reward, next_state, done):
        if len(self.buffer) < self.capacity:
            self.buffer.append(None)
        self.buffer[self.ptr] = (state, action, reward, next_state, done)
        self.ptr = (self.ptr + 1) % self.capacity

    def sample(self, batch_size):
        indices = rng.choice(len(self.buffer), batch_size, replace=False)
        states, actions, rewards, next_states, dones = zip(*[self.buffer[i] for i in indices])
        return (np.array(states), np.array(actions), np.array(rewards, dtype=np.float32),
                np.array(next_states), np.array(dones, dtype=np.float32))

class NeuralQNetwork:
    """The scratch MLP with autograd doing the backward pass. Weights come from
    the same numpy draws, so both lanes start from the same numbers."""

    def __init__(self, state_dim=16, hidden_dim=32, action_dim=4, lr=0.05):
        self.lr = lr
        self.W1 = torch.as_tensor(
            rng.standard_normal((state_dim, hidden_dim)) * np.sqrt(2.0 / state_dim)
        ).requires_grad_(True)
        self.b1 = torch.zeros((1, hidden_dim), dtype=torch.float64, requires_grad=True)
        self.W2 = torch.as_tensor(
            rng.standard_normal((hidden_dim, action_dim)) * np.sqrt(2.0 / hidden_dim)
        ).requires_grad_(True)
        self.b2 = torch.zeros((1, action_dim), dtype=torch.float64, requires_grad=True)

    def forward(self, x):
        x = torch.as_tensor(np.asarray(x, dtype=np.float64))
        self.z1 = x @ self.W1 + self.b1
        self.a1 = torch.relu(self.z1)
        self.q = self.a1 @ self.W2 + self.b2
        return self.q

    def copy_weights_from(self, other):
        with torch.no_grad():
            self.W1.copy_(other.W1)
            self.b1.copy_(other.b1)
            self.W2.copy_(other.W2)
            self.b2.copy_(other.b2)

    def train_step(self, x, actions, targets):
        N = x.shape[0]
        q_pred = self.forward(x)
        chosen = q_pred[torch.arange(N), torch.as_tensor(np.asarray(actions))]
        t = torch.as_tensor(np.asarray(targets, dtype=np.float64))
        loss = 0.5 * torch.mean((chosen - t) ** 2)
        for p in (self.W1, self.b1, self.W2, self.b2):
            p.grad = None
        loss.backward()          # replaces the whole hand-written backward pass
        with torch.no_grad():
            for p in (self.W1, self.b1, self.W2, self.b2):
                p -= self.lr * p.grad
        return float(loss.detach())


In [ ]:
# exports: q_table, first_loss, last_loss
class _GridWorld:
    """Faithful copy of the notebook's GridWorld — the env is a fixture, not a step."""
    def __init__(self, rows=4, cols=4, start=(0, 0), goal=(3, 3),
                 obstacles=((1, 1), (2, 1))):
        self.rows, self.cols, self.start, self.goal = rows, cols, start, goal
        self.obstacles = list(obstacles)
        self.actions = [(-1, 0), (0, 1), (1, 0), (0, -1)]  # Up, Right, Down, Left
        self.reset()

    def reset(self):
        self.current_state = self.start
        return self.current_state

    def step(self, action_idx):
        if self.current_state == self.goal:
            return self.current_state, 0.0, True
        dr, dc = self.actions[action_idx]
        r, c = self.current_state
        nr, nc = r + dr, c + dc
        if 0 <= nr < self.rows and 0 <= nc < self.cols and (nr, nc) not in self.obstacles:
            self.current_state = (nr, nc)
        done = self.current_state == self.goal
        return self.current_state, 10.0 if done else -1.0, done

    def get_all_states(self):
        return [(r, c) for r in range(self.rows) for c in range(self.cols)
                if (r, c) not in self.obstacles]

    def get_transitions(self, state, action_idx):
        if state == self.goal:
            return [(1.0, state, 0.0, True)]
        dr, dc = self.actions[action_idx]
        r, c = state
        nr, nc = r + dr, c + dc
        ns = (nr, nc) if (0 <= nr < self.rows and 0 <= nc < self.cols
                          and (nr, nc) not in self.obstacles) else state
        done = ns == self.goal
        return [(1.0, ns, 10.0 if done else -1.0, done)]

rng = np.random.default_rng(777)
_env_eq = _GridWorld()
_q_eq = NeuralQNetwork(state_dim=16, hidden_dim=32, action_dim=4, lr=0.05)
_tgt_eq = NeuralQNetwork(state_dim=16, hidden_dim=32, action_dim=4, lr=0.05)
_tgt_eq.copy_weights_from(_q_eq)
_buf_eq = ReplayBuffer(capacity=500)
_losses_eq = []
_step_eq = 0
for _ep in range(15):                    # tiny budget: ~460 train steps
    _s = state_to_onehot(_env_eq.reset())
    _done = False
    _t = 0
    while not _done and _t < 40:
        _t += 1
        _step_eq += 1
        _a = int(rng.integers(4))        # pure exploration: no decision reads the net
        _s2_raw, _r, _done = _env_eq.step(_a)
        _s2 = state_to_onehot(_s2_raw)
        _buf_eq.push(_s, _a, _r, _s2, _done)
        _s = _s2
        if len(_buf_eq.buffer) >= 32:
            _bs, _ba, _br, _bs2, _bd = _buf_eq.sample(32)
            _tq = _tgt_eq.forward(_bs2).detach().numpy()
            _y = _br + 0.99 * np.max(_tq, axis=1) * (1.0 - _bd)
            _losses_eq.append(float(_q_eq.train_step(_bs, _ba, _y)))
        if _step_eq % 50 == 0:
            _tgt_eq.copy_weights_from(_q_eq)
q_table = [[float(_x) for _x in
            _q_eq.forward(state_to_onehot(_s).reshape(1, -1)).detach().numpy()[0]]
           for _s in _env_eq.get_all_states()]
first_loss = float(_losses_eq[0])
last_loss = float(_losses_eq[-1])
print("train steps:", len(_losses_eq), "| last loss:", round(last_loss, 5))


In [ ]:
# Autograd against the notebook's hand-written backward pass, on a probe batch.
_px, _pa, _pr, _px2, _pd = _buf_eq.sample(8)
_pt = _pr + 0.99 * np.max(_tgt_eq.forward(_px2).detach().numpy(), axis=1) * (1.0 - _pd)
_xt = torch.as_tensor(np.asarray(_px, dtype=np.float64))
_qp = _q_eq.forward(_xt)
_pn = _xt.shape[0]
_dq = torch.zeros_like(_qp)
for _i in range(_pn):
    _dq[_i, int(_pa[_i])] = (float(_qp[_i, int(_pa[_i])]) - float(_pt[_i])) / _pn
_dW2_hand = _q_eq.a1.detach().T @ _dq
_dz1 = (_dq @ _q_eq.W2.detach().T) * (_q_eq.z1.detach() > 0)
_dW1_hand = _xt.T @ _dz1
_chosen = _qp[torch.arange(_pn), torch.as_tensor(np.asarray(_pa))]
_loss_p = 0.5 * torch.mean((_chosen - torch.as_tensor(np.asarray(_pt, dtype=np.float64))) ** 2)
for _p in (_q_eq.W1, _q_eq.b1, _q_eq.W2, _q_eq.b2):
    _p.grad = None
_loss_p.backward()
assert torch.allclose(_q_eq.W2.grad, _dW2_hand, atol=1e-12), "autograd dW2 == hand dW2"
assert torch.allclose(_q_eq.W1.grad, _dW1_hand, atol=1e-12), "autograd dW1 == hand dW1"

# A target-network sync is an exact copy: identical outputs afterwards.
_tgt_eq.copy_weights_from(_q_eq)
assert torch.equal(_tgt_eq.forward(_px).detach(), _q_eq.forward(_px).detach()), \
    "after a sync the target net answers identically"

# Even this tiny off-policy run learns the move next to the goal.
_states_eq = _env_eq.get_all_states()
assert int(np.argmax(q_table[_states_eq.index((3, 2))])) == 1, \
    "one step left of the goal, Right is the greedy action"

assert max(abs(_v) for _row in q_table for _v in _row) < 50.0, "Q stays in a sane range"


### library

The idiomatic version: `nn.Sequential` of `nn.Linear` + `nn.ReLU`, `gather` for the chosen actions, `optim.SGD` for the step, `load_state_dict` for the target sync — initialised from the very same numpy draws. **What the library adds:** named modules and optimisers with none of the algorithm changed, which is why its Q table matches the scratch one to ~1e-14.

In [ ]:
import numpy as np
import torch

# hints:
# 1. nn.Linear keeps weight as (out, in): copy the numpy draw transposed.
# 2. gather(1, actions) picks q[i, aᵢ] — the loop the scratch writes by hand.
# 3. optim.SGD(lr) is exactly p -= lr·grad here — no momentum, nothing hidden.
# 4. load_state_dict is copy_weights_from with the bookkeeping done for you.
# 5. .double() first, then copy: default nn.Linear tensors are float32.


def state_to_onehot(state, rows=4, cols=4):
    idx = state[0] * cols + state[1]
    onehot = np.zeros(rows * cols, dtype=np.float32)
    onehot[idx] = 1.0
    return onehot

class ReplayBuffer:
    def __init__(self, capacity=2000):
        self.capacity = capacity
        self.buffer = []
        self.ptr = 0

    def push(self, state, action, reward, next_state, done):
        if len(self.buffer) < self.capacity:
            self.buffer.append(None)
        self.buffer[self.ptr] = (state, action, reward, next_state, done)
        self.ptr = (self.ptr + 1) % self.capacity

    def sample(self, batch_size):
        indices = rng.choice(len(self.buffer), batch_size, replace=False)
        states, actions, rewards, next_states, dones = zip(*[self.buffer[i] for i in indices])
        return (np.array(states), np.array(actions), np.array(rewards, dtype=np.float32),
                np.array(next_states), np.array(dones, dtype=np.float32))

class NeuralQNetwork:
    """The same network as an idiomatic nn.Module stack: nn.Linear layers,
    a gather for the chosen actions, optim.SGD for the update — initialised
    from the same numpy draws so all three lanes train the same net."""

    def __init__(self, state_dim=16, hidden_dim=32, action_dim=4, lr=0.05):
        self.lr = lr
        self.net = torch.nn.Sequential(
            torch.nn.Linear(state_dim, hidden_dim),
            torch.nn.ReLU(),
            torch.nn.Linear(hidden_dim, action_dim),
        ).double()
        with torch.no_grad():
            self.net[0].weight.copy_(torch.as_tensor(
                (rng.standard_normal((state_dim, hidden_dim)) * np.sqrt(2.0 / state_dim)).T))
            self.net[0].bias.zero_()
            self.net[2].weight.copy_(torch.as_tensor(
                (rng.standard_normal((hidden_dim, action_dim)) * np.sqrt(2.0 / hidden_dim)).T))
            self.net[2].bias.zero_()
        self.opt = torch.optim.SGD(self.net.parameters(), lr=lr)

    def forward(self, x):
        return self.net(torch.as_tensor(np.asarray(x, dtype=np.float64)))

    def copy_weights_from(self, other):
        self.net.load_state_dict(other.net.state_dict())

    def train_step(self, x, actions, targets):
        q_pred = self.forward(x)
        idx = torch.as_tensor(np.asarray(actions), dtype=torch.long)
        chosen = q_pred.gather(1, idx.unsqueeze(1)).squeeze(1)
        t = torch.as_tensor(np.asarray(targets, dtype=np.float64))
        loss = 0.5 * torch.mean((chosen - t) ** 2)
        self.opt.zero_grad()
        loss.backward()
        self.opt.step()
        return float(loss.detach())


In [ ]:
# exports: q_table, first_loss, last_loss
class _GridWorld:
    """Faithful copy of the notebook's GridWorld — the env is a fixture, not a step."""
    def __init__(self, rows=4, cols=4, start=(0, 0), goal=(3, 3),
                 obstacles=((1, 1), (2, 1))):
        self.rows, self.cols, self.start, self.goal = rows, cols, start, goal
        self.obstacles = list(obstacles)
        self.actions = [(-1, 0), (0, 1), (1, 0), (0, -1)]  # Up, Right, Down, Left
        self.reset()

    def reset(self):
        self.current_state = self.start
        return self.current_state

    def step(self, action_idx):
        if self.current_state == self.goal:
            return self.current_state, 0.0, True
        dr, dc = self.actions[action_idx]
        r, c = self.current_state
        nr, nc = r + dr, c + dc
        if 0 <= nr < self.rows and 0 <= nc < self.cols and (nr, nc) not in self.obstacles:
            self.current_state = (nr, nc)
        done = self.current_state == self.goal
        return self.current_state, 10.0 if done else -1.0, done

    def get_all_states(self):
        return [(r, c) for r in range(self.rows) for c in range(self.cols)
                if (r, c) not in self.obstacles]

    def get_transitions(self, state, action_idx):
        if state == self.goal:
            return [(1.0, state, 0.0, True)]
        dr, dc = self.actions[action_idx]
        r, c = state
        nr, nc = r + dr, c + dc
        ns = (nr, nc) if (0 <= nr < self.rows and 0 <= nc < self.cols
                          and (nr, nc) not in self.obstacles) else state
        done = ns == self.goal
        return [(1.0, ns, 10.0 if done else -1.0, done)]

rng = np.random.default_rng(777)
_env_eq = _GridWorld()
_q_eq = NeuralQNetwork(state_dim=16, hidden_dim=32, action_dim=4, lr=0.05)
_tgt_eq = NeuralQNetwork(state_dim=16, hidden_dim=32, action_dim=4, lr=0.05)
_tgt_eq.copy_weights_from(_q_eq)
_buf_eq = ReplayBuffer(capacity=500)
_losses_eq = []
_step_eq = 0
for _ep in range(15):                    # tiny budget: ~460 train steps
    _s = state_to_onehot(_env_eq.reset())
    _done = False
    _t = 0
    while not _done and _t < 40:
        _t += 1
        _step_eq += 1
        _a = int(rng.integers(4))        # pure exploration: no decision reads the net
        _s2_raw, _r, _done = _env_eq.step(_a)
        _s2 = state_to_onehot(_s2_raw)
        _buf_eq.push(_s, _a, _r, _s2, _done)
        _s = _s2
        if len(_buf_eq.buffer) >= 32:
            _bs, _ba, _br, _bs2, _bd = _buf_eq.sample(32)
            _tq = _tgt_eq.forward(_bs2).detach().numpy()
            _y = _br + 0.99 * np.max(_tq, axis=1) * (1.0 - _bd)
            _losses_eq.append(float(_q_eq.train_step(_bs, _ba, _y)))
        if _step_eq % 50 == 0:
            _tgt_eq.copy_weights_from(_q_eq)
q_table = [[float(_x) for _x in
            _q_eq.forward(state_to_onehot(_s).reshape(1, -1)).detach().numpy()[0]]
           for _s in _env_eq.get_all_states()]
first_loss = float(_losses_eq[0])
last_loss = float(_losses_eq[-1])
print("train steps:", len(_losses_eq), "| last loss:", round(last_loss, 5))


In [ ]:
# gather really is the by-hand indexing loop.
_px, _pa, _pr, _px2, _pd = _buf_eq.sample(8)
_qp = _q_eq.forward(_px).detach()
_g = _qp.gather(1, torch.as_tensor(np.asarray(_pa), dtype=torch.long).unsqueeze(1)).squeeze(1)
assert all(float(_g[_i]) == float(_qp[_i, int(_pa[_i])]) for _i in range(8)), \
    "gather picks exactly q[i, a_i]"

# load_state_dict syncs the target net exactly.
_tgt_eq.copy_weights_from(_q_eq)
_pxs = torch.as_tensor(np.asarray(_px, dtype=np.float64))
assert torch.equal(_tgt_eq.forward(_pxs).detach(), _q_eq.forward(_pxs).detach()), \
    "after a sync the target net answers identically"

# The tiny off-policy run still points the state beside the goal at the goal.
_states_eq = _env_eq.get_all_states()
assert int(np.argmax(q_table[_states_eq.index((3, 2))])) == 1, \
    "one step left of the goal, Right is the greedy action"

assert max(q_table[_states_eq.index((3, 2))]) > max(q_table[_states_eq.index((0, 0))]), \
    "value grows toward the goal"
